In [7]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random

from torch.optim.optimizer import Optimizer
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

In [8]:
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, 
                                       download=True,transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
]))
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
]))

100.0%


Extracting ./data\cifar-10-python.tar.gz to ./data
Files already downloaded and verified


In [ ]:
trainloader = torch.utils.data.DataLoader(trainset, batch_size=100,
                                          shuffle=True, num_workers=0)
testloader = torch.utils.data.DataLoader(testset, batch_size=100,
                                         shuffle=False, num_workers=0)


In [13]:
dataiter = iter(trainloader)
images, labels = next(dataiter)

In [15]:
print("Batch image tensor shape:", images.shape)
print("Batch labels tensor shape:", labels.shape)

Batch image tensor shape: torch.Size([30, 3, 224, 224])
Batch labels tensor shape: torch.Size([30])


In [16]:
num_classes = len(trainset.classes)
print("Number of classes:", num_classes)

Number of classes: 10


In [32]:
class NiNBlock(nn.Module):
    def __init__(self, in_channels,out_channels,k):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=k, 
                      padding=(k-1)//2,stride=1),
            nn.ReLU(inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=1,padding=0,stride=1),
            nn.ReLU(inplace=True)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=1,padding=0,stride=1),
            nn.ReLU(inplace=True)
        )
    def forward(self,x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        return x    


In [53]:
class NiN3(nn.Module):
    def __init__(self):
        super(NiN3,self).__init__()
        self.conv1 = nn.Sequential(
            NiNBlock(3,32,3)
        )
        self.pool1 = nn.MaxPool2d(kernel_size=2, padding=0, stride=2)
        self.conv2 = nn.Sequential(
            NiNBlock(32,64,3)
        )
        self.pool2 = nn.MaxPool2d(kernel_size=2, padding=0, stride=2)
        self.conv3 = nn.Sequential(
            NiNBlock(64,64,3)
        )
        self.conv4 = nn.Conv2d(64,10,kernel_size=1,padding=0,stride=1)
        self.avg_pool  = nn.AdaptiveAvgPool2d((1, 1))
        # self.fc = nn.Linear(128,1)
        self.dummy()
    def dummy(self):
        x = torch.randn(1,3,224,224)
        print(x.shape)
        x = self.conv1(x)
        print(x.shape)
        x = self.pool1(x)
        print(x.shape)
        x = self.conv2(x)
        print(x.shape)
        x = self.pool2(x)
        print(x.shape)
        x = self.conv3(x)
        print(x.shape)
        x = self.conv4(x)
        print(x.shape)
        x = self.avg_pool(x)
        print(x.shape)
             
    def forward(self,x):
        x = self.conv1(x)
        x = self.pool1(x)    
        x = self.conv2(x)
        x = self.pool2(x)
        x = self.conv3(x)
        x = self.conv4(x)
        x = self.avg_pool(x)
        x = x.view(x.size(0),10)

        return x

In [54]:
device = torch.device("cpu")

model = NiN3().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 5

print(f"\nTraining Custom ResNet with {sum(p.numel() for p in model.parameters())} parameters")
print(f"Device: {device}\n")

for epoch in range(num_epochs):
    for i, (images, labels) in enumerate(trainloader):
        images = images.to(device)
        labels = labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if (i+1) % 100 == 0:
            print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}' 
                   .format(epoch+1, num_epochs, i+1, len(trainloader), loss.item()))

model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in testloader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()   
    print('Test Accuracy of the model on the 10000 test images: {} %'.format(100 * correct / total))

torch.save(model.state_dict(), 'model.ckpt')     

torch.Size([1, 3, 224, 224])
torch.Size([1, 32, 224, 224])
torch.Size([1, 32, 112, 112])
torch.Size([1, 64, 112, 112])
torch.Size([1, 64, 56, 56])
torch.Size([1, 64, 56, 56])
torch.Size([1, 10, 56, 56])
torch.Size([1, 10, 1, 1])

Training Custom ResNet with 75722 parameters
Device: cpu

Epoch [1/5], Step [100/1667], Loss: 2.2325
Epoch [1/5], Step [200/1667], Loss: 2.0718
Epoch [1/5], Step [300/1667], Loss: 2.0848
Epoch [1/5], Step [400/1667], Loss: 1.9625
Epoch [1/5], Step [500/1667], Loss: 1.8844
Epoch [1/5], Step [600/1667], Loss: 2.3380
Epoch [1/5], Step [700/1667], Loss: 1.9766
Epoch [1/5], Step [800/1667], Loss: 1.8390
Epoch [1/5], Step [900/1667], Loss: 2.0245
Epoch [1/5], Step [1000/1667], Loss: 1.8971
Epoch [1/5], Step [1100/1667], Loss: 2.1343
Epoch [1/5], Step [1200/1667], Loss: 1.9639
Epoch [1/5], Step [1300/1667], Loss: 2.1553
Epoch [1/5], Step [1400/1667], Loss: 2.0974
Epoch [1/5], Step [1500/1667], Loss: 2.0529
Epoch [1/5], Step [1600/1667], Loss: 2.2285


KeyboardInterrupt: 